Import necessary packages.

In [1]:
import requests
import time
import json

I have discovered two methods of accessing ESPN's API:
1. We understand the structure of the URL based on online resources. For example, the 2016 NBA Finals, Game 7, Cavs vs Warriors can be accessed via,
    https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/summary?region=us&lang=en&contentorigin=espn&event=400878160
    
    We can rewrite this URL more generally as,
    
    https://site.web.api.espn.com/apis/site/v2/sports/{SPORT_NAME}/{LEAGUE_NAME}/summary?region=us&lang=en&contentorigin=espn&event={GAME_ID}
2. Additionally, we can inspect element, select the network tab, and filter traffic by Fetch/XHR. Typically the request we are looking for is 8-15 kB in size but can be
    more definitively recognized by its name $\rightarrow$ `'header?sport=...'`. If we click on this request we can view the Request URL and access the API this way.


In [3]:
# Arsenal vs Bayer Leverkusen. UCL Leg 1, Mar 11 2026
sport = 'soccer'
league = 'uefa.champions'
game_id = '401862578'
custom_url = f'https://site.web.api.espn.com/apis/site/v2/sports/{sport}/{league}/summary?region=us&lang=en&contentorigin=espn&event={game_id}'

request_url = 'https://site.web.api.espn.com/apis/personalized/v2/scoreboard/header?sport=soccer&league=uefa.champions&region=us&lang=en&contentorigin=espn&configuration=STREAM_MENU&platform=web&features=sfb-all%2Ccutl&buyWindow=1m&showAirings=buy%2Clive%2Creplay&showZipLookup=true&tz=America%2FNew_York&postalCode=18938&playabilitySource=playbackId'


Given that we have two methods of accessing the APIs, I will inspect the data from each to ensure they are the same.

In [4]:
custom_url_data = requests.get(custom_url).json()
request_url_data = requests.get(request_url).json()

try:
    with open('custom_url_data.json', 'w') as json_file:
        json.dump(custom_url_data, json_file, indent=2)
    print(f"Custom URL data successfully written")
except IOError as e:
    print(f"Error writing custom url data: {e}")

try:
    with open('request_url_data.json', 'w') as json_file:
        json.dump(request_url_data, json_file, indent=2)
    print(f"Request URL data successfully written")
except IOError as e:
    print(f"Error writing request url data: {e}")


Custom URL data successfully written
Request URL data successfully written


Upon closer inspection I have identified two additional endpoints. We can find these by filtering by Socket. 

In order to use these connections we need to intercept the token using a headless browser that allows us to hook into network requests as the page loads, capturing the WebSocket URL before connecting to it.

In [ ]:
from playwright.async_api import async_playwright
import json
import base64
import zlib

def decode_payload(pl_string):
    compressed = base64.b64decode(pl_string)
    decompressed = zlib.decompress(compressed)
    return json.loads(decompressed)

async def capture_ws():
    async with async_playwright() as p:
        browser = await p.chromium.launch(channel='chrome')
        page = await browser.new_page()

        def on_websocket(ws):
            if 'Fastcast' not in ws.url:
                return
            print(f"Connected to: {ws.url[:80]}...")

            def on_message(frame):
                try:
                    data = json.loads(frame)
                    op = data.get("op")

                    if op == "B":
                        return
                    
                    elif op in ("P", "R"):
                        pl_outer = json.loads(data["pl"])
                        decoded = decode_payload(pl_outer["pl"])
                        print(f"[{op}] ", json.dumps(decoded, indent=2))
                    
                    else:
                        print(json.dumps(data, indent=2))

                except Exception as e:
                    print(f"Error: {e}")
                    print("Raw frame:", frame)

            ws.on("framereceived", on_message)

        page.on("websocket", on_websocket)

        await page.goto('https://www.espn.com/mens-college-basketball/game?gameId=401851496')
        await page.wait_for_timeout(300000)

        await browser.close()

await capture_ws()